In [1]:
# import libraries and paths
import json
import ast
import pandas as pd
import numpy as np
from pathlib import Path

AUDIT  = Path(r"C:\Users\Admin\Endometriosis\endo_project\audit")
SPLITS = Path(r"C:\Users\Admin\Endometriosis\endo_project\data\splits")
SPLITS.mkdir(parents=True, exist_ok=True)

df_files    = pd.read_csv(AUDIT / "file_audit.csv")
df_subjects = pd.read_csv(AUDIT / "subject_level.csv")
df_files["rater"] = df_files["rater"].fillna("single")

print("Files:", len(df_files), "Subjects:", len(df_subjects))

Files: 685 Subjects: 124


In [4]:
# apply
PREFERENCE = ["1", "2", "3", "single"]
PREF_MAP = {r: i for i, r in enumerate(PREFERENCE)}

masks = df_files[df_files["modality"].isin(["em", "ov", "ut"])].copy()

# Assign preference score, then sort, then take first per (subject, centre, modality).
# Note: use a large fallback for unexpected rater values so they sort last.
masks["_pref"] = masks["rater"].map(PREF_MAP)
masks["_pref"] = masks["_pref"].fillna(999).astype(int)

masks = masks.sort_values(
    ["subject", "centre", "modality", "_pref"],
    kind="stable",
)

chosen = (
    masks
    .drop_duplicates(subset=["subject", "centre", "modality"], keep="first")
    .reset_index(drop=True)
)

chosen = chosen[["subject", "centre", "modality", "rater", "file", "path"]].copy()
chosen.columns = [
    "subject", "centre", "structure",
    "chosen_rater", "chosen_file", "chosen_path",
]

chosen.to_csv(AUDIT / "chosen_labels.csv", index=False)

print("Chosen rater distribution (structure × chosen_rater):")
print(chosen.groupby(["structure", "chosen_rater"]).size())
print()
print("Total chosen label files:", len(chosen))
print()
print("Sanity: one row per subject-structure:")
print(chosen.groupby(["subject", "structure"]).size().value_counts())
print(chosen.head())

Chosen rater distribution (structure × chosen_rater):
structure  chosen_rater
em         1.0              9
           2.0             14
           3.0             17
           single          11
ov         1.0             10
           2.0             13
           3.0             20
           single          58
ut         1.0              9
           2.0             18
           3.0             22
           single          62
dtype: int64

Total chosen label files: 263

Sanity: one row per subject-structure:
1    263
Name: count, dtype: int64
  subject  centre structure chosen_rater          chosen_file  \
0  D1-000  D1_MHS        em          1.0  D1-000_em_r1.nii.gz   
1  D1-000  D1_MHS        ov          1.0  D1-000_ov_r1.nii.gz   
2  D1-000  D1_MHS        ut          1.0  D1-000_ut_r1.nii.gz   
3  D1-001  D1_MHS        em          1.0  D1-001_em_r1.nii.gz   
4  D1-001  D1_MHS        ov          1.0  D1-001_ov_r1.nii.gz   

                                         chosen_path

In [5]:
# master subject table
df = df_subjects.copy()
df["sequences"] = df["sequences"].apply(
    lambda s: ast.literal_eval(s) if isinstance(s, str) else s
)
df["usable"] = (df["n_sequences"] > 0) & df["has_ut"]  # exclude zero-seq subjects

# Drop non-usable subjects from all future splits
usable = df[df["usable"]].copy()
print("Usable subjects:", len(usable), "of", len(df))
print(usable.groupby(["centre","has_em"]).size())

usable.to_csv(AUDIT / "subject_level_final.csv", index=False)

Usable subjects: 108 of 124
centre   has_em
D1_MHS   False     10
         True      38
D2_TCPW  False     54
         True       6
dtype: int64


In [6]:
# stratification
usable["stratum"] = (
    usable["centre"] + "_em" + usable["has_em"].astype(int).astype(str)
)
print("Strata counts:")
print(usable["stratum"].value_counts())

Strata counts:
stratum
D2_TCPW_em0    54
D1_MHS_em1     38
D1_MHS_em0     10
D2_TCPW_em1     6
Name: count, dtype: int64


In [7]:
# street 5-fold split
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = []
for i, (tr, va) in enumerate(skf.split(usable, usable["stratum"])):
    tr_df, va_df = usable.iloc[tr], usable.iloc[va]
    folds.append({
        "fold": i,
        "train": sorted(tr_df["subject"].tolist()),
        "val":   sorted(va_df["subject"].tolist()),
        "train_centres": tr_df["centre"].value_counts().to_dict(),
        "val_centres":   va_df["centre"].value_counts().to_dict(),
        "train_has_em":  int(tr_df["has_em"].sum()),
        "val_has_em":    int(va_df["has_em"].sum()),
    })

with open(SPLITS / "cv5_folds.json", "w") as f:
    json.dump(folds, f, indent=2)

for f in folds:
    print(f"Fold {f['fold']}: train={len(f['train'])} val={len(f['val'])} "
          f"val_centres={f['val_centres']} val_has_em={f['val_has_em']}")

Fold 0: train=86 val=22 val_centres={'D2_TCPW': 12, 'D1_MHS': 10} val_has_em=9
Fold 1: train=86 val=22 val_centres={'D2_TCPW': 12, 'D1_MHS': 10} val_has_em=9
Fold 2: train=86 val=22 val_centres={'D2_TCPW': 12, 'D1_MHS': 10} val_has_em=10
Fold 3: train=87 val=21 val_centres={'D2_TCPW': 12, 'D1_MHS': 9} val_has_em=8
Fold 4: train=87 val=21 val_centres={'D2_TCPW': 12, 'D1_MHS': 9} val_has_em=8


In [8]:
# freeze leave one centre out
loco = {}
for test_centre in sorted(usable["centre"].unique()):
    test = usable[usable["centre"] == test_centre]
    train = usable[usable["centre"] != test_centre]
    loco[test_centre] = {
        "train": sorted(train["subject"].tolist()),
        "test":  sorted(test["subject"].tolist()),
        "train_centres": train["centre"].value_counts().to_dict(),
        "test_centres":  test["centre"].value_counts().to_dict(),
        "train_has_em":  int(train["has_em"].sum()),
        "test_has_em":   int(test["has_em"].sum()),
    }
    print(f"Held out {test_centre}: train={len(train)} test={len(test)} "
          f"test_has_em={int(test['has_em'].sum())}")

with open(SPLITS / "leave_one_centre_out.json", "w") as f:
    json.dump(loco, f, indent=2)

Held out D1_MHS: train=60 test=48 test_has_em=38
Held out D2_TCPW: train=48 test=60 test_has_em=6


In [9]:
# freeze matched subsets
def seq_set(s):
    return set(s) if isinstance(s, list) else set()

all4          = usable[usable["sequences"].apply(lambda s: seq_set(s) == {"T1","T1FS","T2","T2FS"})]
t1fs_t2       = usable[usable["sequences"].apply(lambda s: {"T1FS","T2"}.issubset(seq_set(s)))]
t1_t1fs_t2fs  = usable[usable["sequences"].apply(lambda s: seq_set(s) == {"T1","T1FS","T2FS"})]

print("all4:", len(all4), "  centres:", all4["centre"].value_counts().to_dict())
print("t1fs_t2:", len(t1fs_t2), "  centres:", t1fs_t2["centre"].value_counts().to_dict())
print("t1_t1fs_t2fs:", len(t1_t1fs_t2fs), "  centres:", t1_t1fs_t2fs["centre"].value_counts().to_dict())

matched = {
    "all4":         sorted(all4["subject"].tolist()),
    "t1fs_t2":      sorted(t1fs_t2["subject"].tolist()),
    "t1_t1fs_t2fs": sorted(t1_t1fs_t2fs["subject"].tolist()),
    "note": (
        "all4 and t1_t1fs_t2fs are D2-only. "
        "t1fs_t2 is the primary cross-centre matched subset."
    ),
}
with open(SPLITS / "matched_subsets.json", "w") as f:
    json.dump(matched, f, indent=2)
print("\nSaved matched_subsets.json")

all4: 31   centres: {'D2_TCPW': 31}
t1fs_t2: 69   centres: {'D1_MHS': 37, 'D2_TCPW': 32}
t1_t1fs_t2fs: 21   centres: {'D2_TCPW': 21}

Saved matched_subsets.json


In [10]:
# verify
print("Frozen split files:")
for p in sorted(SPLITS.iterdir()):
    print(" -", p.name)

Frozen split files:
 - cv5_folds.json
 - leave_one_centre_out.json
 - matched_subsets.json
